In [ ]:
from pathlib import Path

import numpy as np
import pulp
from IPython.display import display
from PIL import Image
from scipy.cluster import hierarchy
from scipy.spatial.distance import squareform


In [ ]:
ROWS = 11  # 横切数量
COLS = 19  # 纵切数量

CHINESE_CHARACTER_HEIGHT = 40  # 中文字高
CHINESE_ROW_SPACE = 68  # 中文行距
ENGLISH_MIDDLE_LINE_HEIGHT = 24  # 英文中线高
ENGLISH_ROW_SPACE = 64  # 英文行距


In [ ]:
# 从附件中读取图片，并转换为 numpy 矩阵

def read_image(path):
    img = Image.open(path).convert('L')
    mat = np.array(img)
    return mat


def read_images(folder: Path, num: int, ab: bool):
    """从 folder 中读取图片"""
    if not ab:
        images = [read_image(folder / f"{i:03d}.bmp") for i in range(num)]
    else:
        images = [[read_image(folder / f"{i:03d}{j}.bmp") for j in "ab"] for i in range(num)]
    return np.array(images)


root_folder = Path("附件")
images_1 = read_images(root_folder / "附件1", COLS, ab=False)
images_2 = read_images(root_folder / "附件2", COLS, ab=False)
images_3 = read_images(root_folder / "附件3", ROWS * COLS, ab=False)
images_4 = read_images(root_folder / "附件4", ROWS * COLS, ab=False)
images_5ab = read_images(root_folder / "附件5", ROWS * COLS, ab=True)


In [ ]:
# 一些辅助函数

def parse_range_string(range_str: str):
    """把形如 `"0-3,6-5,9"` 的字符串转换成列表 `[0, 1, 2, 3, 6, 5, 9]`"""
    try:
        result = []
        range_str = range_str.replace(' ', '')
        ranges = range_str.split(',')
        for r in ranges:
            if '-' not in r:
                result.append(int(r))
            else:
                start, end = map(int, r.split('-'))
                if start > end:
                    result.extend(range(start, end - 1, -1))
                else:
                    result.extend(range(start, end + 1))
        return result

    except Exception as e:
        raise ValueError(f"Invalid range string: {range_str}") from e


def parse_direction_string(direction_string: str):
    """把形如 `"001110"` 的字符串转换成列表 `[False, False, True, True, True, False]`"""
    return [c == '1' for c in direction_string]


In [ ]:
# 旅行商问题

def solve_tsp(distance_matrix):
    num_cities = len(distance_matrix)
    prob = pulp.LpProblem("TSP", pulp.LpMinimize)

    # 决策变量
    x = pulp.LpVariable.dicts("x", ((i, j) for i in range(num_cities) for j in range(num_cities)), cat='Binary')

    # 目标函数
    prob += pulp.lpSum(distance_matrix[i][j] * x[i, j] for i in range(num_cities) for j in range(num_cities))

    # 约束条件
    for i in range(num_cities):
        prob += pulp.lpSum(x[i, j] for j in range(num_cities) if i != j) == 1
        prob += pulp.lpSum(x[j, i] for j in range(num_cities) if i != j) == 1

    u = pulp.LpVariable.dicts("u", range(num_cities), lowBound=0, upBound=num_cities - 1, cat='Integer')
    for i in range(1, num_cities):
        for j in range(1, num_cities):
            if i != j:
                prob += u[i] - u[j] + (num_cities - 1) * x[i, j] <= num_cities - 2

    prob.solve()

    # 提取解
    tour = []
    current_city = 0
    while True:
        tour.append(current_city)
        next_city = None
        for j in range(num_cities):
            if x[current_city, j].value() == 1:
                next_city = j
                break
        if next_city is None:
            break
        current_city = next_city
        if current_city == 0:
            break

    return tour


In [ ]:
# 拼合图片的函数

def combine_images(images, *, direction="horizontal", seam=False):
    """拼合图片，可选择拼合的方向，以及是否显示接缝"""
    if seam and direction == "horizontal":
        for image in images:
            image[:, 0] = 0
    if seam and direction == "vertical":
        for image in images:
            image[0, :] = 0
    if direction == "horizontal":
        combined_image = np.concatenate(images, axis=1)
    else:
        combined_image = np.concatenate(images, axis=0)
    return combined_image


In [ ]:
# 问题 1

def edge_difference(image0, image1):
    """计算 image0 的右边缘与 image1 的左边缘的差异"""
    image0_right_edge = image0[:, -1].astype(int)  # 必须转换为 int，否则两个 uint8 相减会溢出
    image1_left_edge = image1[:, 0].astype(int)
    return np.sum(np.abs(image0_right_edge - image1_left_edge))


def compute_distance_matrix(images, difference_function):
    """给定距离函数，计算图片之间的距离矩阵"""
    num = len(images)
    distance_matrix = np.zeros((num, num))
    for i, image_i in enumerate(images):
        for j, image_j in enumerate(images):
            distance_matrix[i, j] = difference_function(image_i, image_j)
    return distance_matrix


def find_left_image(images):
    """寻找 images 中左边缘最白的图片"""
    return np.argmax(np.sum(images[:, :, :10], axis=(1, 2)))


def problem_1_tour(images):
    """求解问题 1 的 TSP 问题，返回图片的顺序"""
    distance_matrix = compute_distance_matrix(images, edge_difference)
    tour = solve_tsp(distance_matrix)
    left_image_index = find_left_image(images)
    left_image_in_tour = tour.index(left_image_index)
    tour = tour[left_image_in_tour:] + tour[:left_image_in_tour]
    return np.array(tour)


def problem_1(images):
    """问题 1"""
    tour = problem_1_tour(images)
    print(f"图片顺序：{tour}")
    display(Image.fromarray(combine_images(images[tour], seam=True)))


In [ ]:
# 问题 2

def dilate(vector, radius):
    kernel = np.ones(2 * radius + 1, dtype=np.bool)
    return np.convolve(vector, kernel, "same")


def erode(vector, radius):
    return ~dilate(~vector, radius)


def morph_open(vector, radius):
    """开运算，去除小的 True 游程"""
    return dilate(erode(vector, radius), radius)


def find_False_runs(vector):
    """寻找 False 游程"""
    changes = np.diff(vector, prepend=np.True_, append=np.True_)
    (change_indices,) = np.where(changes)
    start_indices = change_indices[::2]
    end_indices = change_indices[1::2]
    return start_indices, end_indices


def chinese_barcode(image):
    """对于中文图片，考察每一行是否有黑色像素，形成条形码，并补充空白行"""
    vector = np.all(image, axis=1)  # 必须全为白色才把这一行认为是白色
    vector = morph_open(vector, 5)  # 去除小的白色区域
    black_runs_start, black_runs_end = find_False_runs(vector)  # 黑色区域
    if black_runs_start[0] > 0:
        first_black_run_middle = (black_runs_start[0] + black_runs_end[0]) // 2
    else:
        first_black_run_middle = (black_runs_start[1] + black_runs_end[1]) // 2

    # 补充空白行
    for i in range(-2, 3):
        y = first_black_run_middle + i * CHINESE_ROW_SPACE
        if not (0 <= y < image.shape[0]):
            continue
        if not vector[y]:  # 如果已经是黑色，则无需补充
            continue
        y0 = np.clip(y - CHINESE_CHARACTER_HEIGHT // 2, 0, image.shape[0])
        y1 = np.clip(y + CHINESE_CHARACTER_HEIGHT // 2, 0, image.shape[0])
        vector[y0:y1] = 0
    return (vector * 255).astype(np.uint8), first_black_run_middle


def english_barcode(image):
    """对于英文图片，取三行四线的中间行，形成条形码"""
    vector = np.sum(255 - image, axis=1)  # vector[i] 越大，说明第 i 行越黑
    best_sum = -np.inf
    for first_line_top in range(-ENGLISH_ROW_SPACE + 1, 1):
        sum_vector = 0
        for line_top in range(first_line_top, image.shape[0], ENGLISH_ROW_SPACE):
            line_bottom = line_top + ENGLISH_MIDDLE_LINE_HEIGHT
            line_top = np.clip(line_top, 0, image.shape[0])
            line_bottom = np.clip(line_bottom, 0, image.shape[0])
            sum_vector += np.sum(vector[line_top:line_bottom])
        if sum_vector > best_sum:
            best_sum = sum_vector
            best_first_line_top = first_line_top

    result = np.full(image.shape[0], 255, dtype=np.uint8)
    for line_top in range(best_first_line_top, image.shape[0], ENGLISH_ROW_SPACE):
        line_bottom = line_top + ENGLISH_MIDDLE_LINE_HEIGHT
        line_top = np.clip(line_top, 0, image.shape[0])
        line_bottom = np.clip(line_bottom, 0, image.shape[0])
        result[line_top:line_bottom] = 0
    return result, best_first_line_top


def barcode_difference(barcode_0, barcode_1):
    """计算 image0 的条形码与 image1 的条形码的差异"""
    return np.sum(np.abs(barcode_0.astype(int) - barcode_1.astype(int)))


def problem_2_clustering(barcodes, num_clusters):
    """求解问题 2 的聚类问题"""
    distance_matrix = compute_distance_matrix(barcodes, barcode_difference)
    y = squareform(distance_matrix)
    Z = hierarchy.linkage(y, method="single")
    result = hierarchy.fcluster(Z, num_clusters, criterion="maxclust")
    clusters = []
    for cluster_id in range(1, num_clusters + 1):
        (indices,) = np.where(result == cluster_id)
        clusters.append(indices)
    return clusters


def problem_2_cluster_human_intervention(images, clusters, barcodes):
    """聚类人工干预"""
    for i, cluster in enumerate(clusters):
        print(f"Cluster {i}: {cluster}")
        if len(cluster) == 0:
            continue

        parts = []
        for index in cluster:
            parts.append(images[index])
            parts.append(barcodes[index].reshape(-1, 1) * np.ones((1, 32), dtype=np.bool))
        display(Image.fromarray(combine_images(parts, seam=False)))


def problem_2_combine_line(images, clusters):
    """行内拼接"""
    return [problem_1_tour(images[cluster]) for cluster in clusters]


def problem_2_combine_line_human_intervention(images, clusters, tours):
    """行内拼接人工干预"""
    for i, (cluster, tour) in enumerate(zip(clusters, tours)):
        print(f"Cluster {i}: {cluster[tour]}")
        display(Image.fromarray(combine_images(images[cluster][tour], seam=True)))


def distance_between_lines_chinese(image0, image1):
    """假设同一行的纸片已经行内拼接，计算 image0 在上，image1 在下拼接时的“合适程度”"""
    image0_first_line_middle = chinese_barcode(image0)[1]
    image1_first_line_middle = chinese_barcode(image1)[1]
    x = (image1_first_line_middle + image0.shape[0] - image0_first_line_middle) / CHINESE_ROW_SPACE
    return abs(x - round(x))


def distance_between_lines_english(image0, image1):
    """假设同一行的纸片已经行内拼接，计算 image0 在上，image1 在下拼接时的“合适程度”"""
    image0_first_line_top = english_barcode(image0)[1]
    image1_first_line_top = english_barcode(image1)[1]
    x = (image1_first_line_top + image0.shape[0] - image0_first_line_top) / ENGLISH_ROW_SPACE
    return abs(x - round(x)) * ENGLISH_ROW_SPACE + np.sum(np.abs(image0[-1].astype(int) - image1[0].astype(int))) / 10


def find_upper_image(images):
    """寻找 images 中上边缘最白的图片"""
    return np.argmax(np.sum(images[:, 0:30, :], axis=(1, 2)))


def problem_2_lines_tour(images, distance_function):
    """求解问题行间拼接 2 的 TSP 问题，返回图片的顺序"""
    distance_matrix = compute_distance_matrix(images, distance_function)
    tour = solve_tsp(distance_matrix)
    upper_image_index = find_upper_image(images)
    upper_image_in_tour = tour.index(upper_image_index)
    tour = tour[upper_image_in_tour:] + tour[:upper_image_in_tour]
    return tour


In [ ]:
# 问题 3

def problem_3_row_difference(barcode_0ab, barcode_1ab):
    barcode_0a, barcode_0b = barcode_0ab
    barcode_1a, barcode_1b = barcode_1ab
    aa = np.sum(barcode_0a ^ barcode_1a) + np.sum(barcode_0b ^ barcode_1b)
    ab = np.sum(barcode_0a ^ barcode_1b) + np.sum(barcode_0b ^ barcode_1a)
    return min(aa, ab), aa <= ab


# def problem_3_edge_difference(image_0ab, image_1ab):
#     image_0a, image_0b = image_0ab
#     image_1a, image_1b = image_1ab
#     aa = edge_difference(image_0a, image_1a) + edge_difference(image_1b, image_0b)
#     ab = edge_difference(image_0a, image_1b) + edge_difference(image_1a, image_0b)
#     ba = edge_difference(image_0b, image_1a) + edge_difference(image_1b, image_0a)
#     bb = edge_difference(image_0b, image_1b) + edge_difference(image_1a, image_0a)
#     return min(aa, ab, ba, bb), np.argmin([aa, ab, ba, bb])


def problem_3_clustering(barcodes, num_clusters):
    """求解问题 3 的聚类问题"""
    n = len(barcodes)
    distance_matrix = np.zeros((n, n))
    direction_matrix = np.zeros((n, n), dtype=np.bool)
    for i in range(n):
        for j in range(n):
            distance_matrix[i, j], direction_matrix[i, j] = problem_3_row_difference(barcodes[i], barcodes[j])

    y = squareform(distance_matrix)
    Z = hierarchy.linkage(y, method="single")
    result = hierarchy.fcluster(Z, num_clusters, criterion="maxclust")
    clusters = []
    for cluster_id in range(1, num_clusters + 1):
        indices = np.where(result == cluster_id)[0]
        clusters.append(indices)

    direction_lines = []
    for cluster in clusters:
        direction_lines.append(np.array([direction_matrix[cluster[0], x] for x in cluster]))
    return clusters, direction_lines


def problem_3_cluster_human_intervention(images_ab, clusters, direction_lines, barcodes_ab):
    """问题 3 聚类人工干预"""
    # TODO: 重构
    # images_a, images_b = images_ab[:, 0], images_ab[:, 1]
    # barcodes_a, barcodes_b = barcodes_ab[:, 0], barcodes_ab[:, 1]
    # for i, (cluster, direction_line) in enumerate(zip(clusters, direction_lines)):
    #     print(f"Cluster {i}: {cluster}")
    #     if len(cluster) == 0:
    #         continue

    #     parts_a = []
    #     parts_b = []
    #     for index, direction in zip(cluster, direction_line):
    #         if direction:
    #             parts_a.append(images_a[index])
    #             parts_a.append(barcodes_a[index].reshape(-1, 1) * np.ones((1, 32), dtype=np.bool))
    #             parts_b.append(images_b[index])
    #             parts_b.append(barcodes_b[index].reshape(-1, 1) * np.ones((1, 32), dtype=np.bool))
    #         else:
    #             parts_a.append(images_b[index])
    #             parts_a.append(barcodes_b[index].reshape(-1, 1) * np.ones((1, 32), dtype=np.bool))
    #             parts_b.append(images_a[index])
    #             parts_b.append(barcodes_a[index].reshape(-1, 1) * np.ones((1, 32), dtype=np.bool))
    #     display(Image.fromarray(combine_images(parts_a, seam=False)))
    #     display(Image.fromarray(combine_images(parts_b, seam=False)))
    for i, (cluster, direction_line) in enumerate(zip(clusters, direction_lines)):
        print(f"Cluster {i}: {cluster}")
        if len(cluster) == 0:
            continue

        n = len(cluster)
        images_a = images_ab[cluster][np.arange(n), direction_line.astype(int)]
        barcodes_a = barcodes_ab[cluster][np.arange(n), direction_line.astype(int)]
        parts_a = []
        for image, barcode in zip(images_a, barcodes_a):
            parts_a.append(image)
            parts_a.append(barcode.reshape(-1, 1) * np.ones((1, 32), dtype=np.bool))

        images_b = images_ab[cluster][np.arange(n), 1 - direction_line]
        barcodes_b = barcodes_ab[cluster][np.arange(n), 1 - direction_line]
        parts_b = []
        for image, barcode in zip(images_b, barcodes_b):
            parts_b.append(image)
            parts_b.append(barcode.reshape(-1, 1) * np.ones((1, 32), dtype=np.bool))

        display(Image.fromarray(combine_images(parts_a, seam=False)))
        display(Image.fromarray(combine_images(parts_b, seam=False)))


def problem_3_tour(images_ab):
    """行内拼接"""
    if len(images_ab) == 0:
        return np.array([]), np.array([])

    n = len(images_ab)
    # distance_matrix = np.zeros((n, n))
    # direction_matrix = np.zeros((n, n), dtype=int)
    distance_matrix_aa = np.zeros((n, n))
    distance_matrix_ab = np.zeros((n, n))
    distance_matrix_ba = np.zeros((n, n))
    distance_matrix_bb = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            # distance_matrix[i, j], direction_matrix[i, j] = problem_3_edge_difference(images_ab[i], images_ab[j])
            distance_matrix_aa[i, j] = edge_difference(images_ab[i][0], images_ab[j][0]) + edge_difference(images_ab[j][1], images_ab[i][1])
            distance_matrix_ab[i, j] = edge_difference(images_ab[i][0], images_ab[j][1]) + edge_difference(images_ab[j][0], images_ab[i][1])
            distance_matrix_ba[i, j] = edge_difference(images_ab[i][1], images_ab[j][0]) + edge_difference(images_ab[j][1], images_ab[i][0])
            distance_matrix_bb[i, j] = edge_difference(images_ab[i][1], images_ab[j][1]) + edge_difference(images_ab[j][0], images_ab[i][0])
    # print(distance_matrix)
    # tour = solve_tsp(distance_matrix)
    # directions = [False]
    # for i in range(1, n):
    #     last_direction = directions[-1]
    #     direction = direction_matrix[tour[i-1], tour[i]]
    #     if not (not last_direction and direction in (0, 1) or last_direction and direction in (2, 3)):
    #         print(f"Warning: direction mismatch at {i}")
    #     directions.append(direction_matrix[tour[i-1], tour[i]] in (1, 3))
    # return np.array(tour), np.array(directions)
    best_target = np.inf
    for start_index in range(n):
        tour = [start_index]
        directions = [False]
        left_indexes = set(range(0, n)) - {start_index}
        target = 0
        for i in list(range(1, n)) + [0]:
            last_index = tour[-1] if i > 0 else start_index
            last_direction = directions[-1] if i > 0 else False
            best_distance = np.inf
            for j in left_indexes if i > 0 else {start_index}:
                for direction in (False, True):
                    if not last_direction and not direction:
                        distance = distance_matrix_aa[last_index, j]
                    elif not last_direction and direction:
                        distance = distance_matrix_ab[last_index, j]
                    elif last_direction and not direction:
                        distance = distance_matrix_ba[last_index, j]
                    else:
                        distance = distance_matrix_bb[last_index, j]
                    if distance < best_distance:
                        best_distance = distance
                        best_index = j
                        best_direction = direction
            if i > 0:
                tour.append(best_index)
                directions.append(best_direction)
                left_indexes.remove(best_index)
            target += best_distance
        if target < best_target:
            best_target = target
            best_tour = tour
            best_directions = directions

    first_left_likely = np.argmax([np.sum(image_ab[int(direction)][:, :10]) + np.sum(image_ab[1 - direction][:, -10:]) for image_ab, direction in zip(images_ab[best_tour], best_directions)])
    best_tour = best_tour[first_left_likely:] + best_tour[:first_left_likely]
    best_directions = best_directions[first_left_likely:] + best_directions[:first_left_likely]

    return np.array(best_tour), np.array(best_directions)


def problem_3_combine_line(images_ab, clusters):
    """行内拼接"""
    tours_and_directions = [problem_3_tour(images_ab[cluster]) for cluster in clusters]
    return [tour for tour, _ in tours_and_directions], [direction for _, direction in tours_and_directions]


def problem_3_combine_line_human_intervention(images_ab, clusters, tours, directions) -> None:
    """行内拼接人工干预"""
    for i, (cluster, tour, direction_line) in enumerate(zip(clusters, tours, directions)):
        print(f"Cluster {i}: {cluster}")
        if len(cluster) == 0:
            continue
        n = len(cluster)
        display(Image.fromarray(combine_images(images_ab[cluster][tour][np.arange(n), direction_line.astype(int)], seam=True)))
        display(Image.fromarray(combine_images(images_ab[cluster][tour][np.arange(n), 1 - direction_line][::-1], seam=True)))


def distance_between_lines_english_ab(image_0ab, image_1ab):
    image_0a, image_0b = image_0ab
    image_1a, image_1b = image_1ab
    aa = distance_between_lines_english(image_0a, image_1a) + distance_between_lines_english(image_0b, image_1b)
    ab = distance_between_lines_english(image_0a, image_1b) + distance_between_lines_english(image_0b, image_1a)
    return min(aa, ab), np.bool(np.argmin([aa, ab]))


def problem_3_lines_tour(images_ab):
    """求解问题行间拼接 3 的 TSP 问题，返回图片的顺序"""
    n = len(images_ab)
    distance_matrix = np.zeros((n, n))
    direction_matrix = np.zeros((n, n), dtype=np.bool)
    for i in range(n):
        for j in range(n):
            distance_matrix[i, j], direction_matrix[i, j] = distance_between_lines_english_ab(images_ab[i], images_ab[j])
    tour = solve_tsp(distance_matrix)
    directions = [False]
    for i in range(1, n):
        last_direction = directions[-1]
        direction = direction_matrix[tour[i-1], tour[i]]
        directions.append(last_direction ^ direction)  # type: ignore

    first_top_likely = np.argmax([np.sum(image_ab[int(direction)][:30, :]) + np.sum(image_ab[1 - direction][:30, :]) for image_ab, direction in zip(images_ab[tour], directions)])
    tour = tour[first_top_likely:] + tour[:first_top_likely]
    directions = directions[first_top_likely:] + directions[:first_top_likely]

    return np.array(tour), np.array(directions)


def problem_3_tour_human_intervention(images_ab, tour, direction) -> None:
    """行间拼接人工干预"""
    display(Image.fromarray(combine_images(images_ab[tour][np.arange(ROWS), direction.astype(int)], direction="vertical", seam=True)))
    display(Image.fromarray(combine_images(images_ab[tour][np.arange(ROWS), 1 - direction], direction="vertical", seam=True)))


In [ ]:
# 问题 1 附件 1

problem_1(images_1)


In [ ]:
# 问题 1 附件 2

problem_1(images_2)


In [ ]:
# 问题 2 附件 3 尝试按行聚类

barcodes_3 = np.array([chinese_barcode(image)[0] for image in images_3])
clusters_3 = problem_2_clustering(barcodes_3, ROWS)
problem_2_cluster_human_intervention(images_3, clusters_3, barcodes_3)


In [ ]:
# 问题 2 附件 3 按行聚类人工干预

# 实际上无需人工干预


In [ ]:
# 问题 2 附件 3 尝试行内拼接

tours_3 = problem_2_combine_line(images_3, clusters_3)
problem_2_combine_line_human_intervention(images_3, clusters_3, tours_3)


In [ ]:
# 问题 2 附件 3 行内拼接人工干预

# 实际上无需人工干预


In [ ]:
# 问题 2 附件 3 行间拼接

line_images = np.array([combine_images(images_3[cluster][tour])
                         for cluster, tour in zip(clusters_3, tours_3)])
tour_3 = problem_2_lines_tour(line_images, distance_between_lines_chinese)
Image.fromarray(combine_images(line_images[tour_3], seam=True, direction="vertical"))


In [ ]:
# 问题 2 附件 4 尝试按行聚类

barcodes_4 = [english_barcode(image)[0] for image in images_4]
clusters_4 = problem_2_clustering(barcodes_4, ROWS)
problem_2_cluster_human_intervention(images_4, clusters_4, barcodes_4)


In [ ]:
# 问题 2 附件 4 尝试行内拼接

tours_4 = problem_2_combine_line(images_4, clusters_4[:7])
problem_2_combine_line_human_intervention(images_4, clusters_4[:7], tours_4)


In [ ]:
# 问题 2 附件 4 按行聚类人工干预

cluster_index = 1
old_clusters_4 = clusters_4.copy()

clusters_4[0] = old_clusters_4[0][tours_4[0][parse_range_string("0-18")]]
clusters_4[7] = old_clusters_4[0][tours_4[0][parse_range_string("19-37")]]

clusters_4[1] = old_clusters_4[1][tours_4[1][parse_range_string("15-33")]]
clusters_4[8] = old_clusters_4[1][tours_4[1][parse_range_string("0-14,34-37")]]

clusters_4[2] = old_clusters_4[2][tours_4[2][parse_range_string("0-18")]]
clusters_4[9] = old_clusters_4[2][tours_4[2][parse_range_string("19-37")]]

clusters_4[4] = old_clusters_4[4][tours_4[4][parse_range_string("0-18")]]
clusters_4[10] = old_clusters_4[4][tours_4[4][parse_range_string("19-37")]]


In [ ]:
# 问题 2 附件 4 尝试行内拼接

tours_4 = problem_2_combine_line(images_4, clusters_4)
problem_2_combine_line_human_intervention(images_4, clusters_4, tours_4)


In [ ]:
# 问题 2 附件 4 行内拼接人工干预

tours_4[6] = tours_4[6][parse_range_string("0-4,9-14,6-8,5,15-18")]

problem_2_combine_line_human_intervention(images_4, clusters_4, tours_4)


In [ ]:
# 问题 2 行间拼接

line_images = np.array([combine_images(images_4[cluster][tour])
                        for cluster, tour in zip(clusters_4, tours_4)])
tour_4 = problem_2_lines_tour(line_images, distance_between_lines_english)
Image.fromarray(combine_images(line_images[tour_4], seam=True, direction="vertical"))


In [ ]:
# 问题 3 附件 5 尝试按行聚类

barcodes_5ab = np.array([[english_barcode(image)[0] for image in image_ab] for image_ab in images_5ab])
clusters_5, directions_5 = problem_3_clustering(barcodes_5ab, ROWS)

problem_3_cluster_human_intervention(images_5ab, clusters_5, directions_5, barcodes_5ab)


In [ ]:
# 问题 3 附件 5 尝试行内拼接

tours_5, directions_5 = problem_3_combine_line(images_5ab, clusters_5)
problem_3_combine_line_human_intervention(images_5ab, clusters_5, tours_5, directions_5)


In [ ]:
# 问题 3 附件 5 按行聚类人工干预

old_clusters_5 = clusters_5.copy()

clusters_5[0] = old_clusters_5[0][tours_5[0]][parse_range_string("0-18")]
clusters_5[8] = old_clusters_5[0][tours_5[0]][parse_range_string("19-37")]

clusters_5[1] = old_clusters_5[1][tours_5[1]][parse_range_string("0-14,34-37")]
clusters_5[9] = old_clusters_5[1][tours_5[1]][parse_range_string("15-33")]

clusters_5[4] = old_clusters_5[4][tours_5[4]][parse_range_string("0-12,28-33")]
clusters_5[10] = old_clusters_5[4][tours_5[4]][parse_range_string("13-27,34-37")]


In [ ]:
# 问题 3 附件 5 尝试行内拼接

tours_5, directions_5 = problem_3_combine_line(images_5ab, clusters_5)
problem_3_combine_line_human_intervention(images_5ab, clusters_5, tours_5, directions_5)


In [ ]:
# 问题 3 附件 5 行内拼接人工干预

new_tour = parse_range_string("1-9,16-18,14-10,15,0")
new_direction = parse_direction_string("0111111111000001111")
tours_5[4] = tours_5[4][new_tour]
directions_5[4] = ~(directions_5[4] ^ new_direction)[new_tour]

new_tour = parse_range_string("6-18,5-0")
new_direction = parse_direction_string("0000001111111111111")
tours_5[5] = tours_5[5][new_tour]
directions_5[5] = ~(directions_5[5] ^ new_direction)[new_tour]

new_tour = parse_range_string("0-5,13-6,14-18")
new_direction = parse_direction_string("1111110000000011111")
tours_5[7] = tours_5[7][new_tour]
directions_5[7] = ~(directions_5[7] ^ new_direction)[new_tour]

new_tour = parse_range_string("0-4,17-5,18")
new_direction = parse_direction_string("1111100000000000001")
tours_5[9] = tours_5[9][new_tour]
directions_5[9] = ~(directions_5[9] ^ new_direction)[new_tour]

problem_3_combine_line_human_intervention(images_5ab, clusters_5, tours_5, directions_5)


In [ ]:
# 问题 3 附件 5 尝试行间拼接

line_images_ab = np.array([
    (combine_images(images_5ab[cluster][tour][np.arange(COLS), direction_line.astype(int)]),
     combine_images(images_5ab[cluster][tour][np.arange(COLS), 1 - direction_line][::-1]))
    for cluster, tour, direction_line in zip(clusters_5, tours_5, directions_5)
])
tour_5, direction_5 = problem_3_lines_tour(line_images_ab)
problem_3_tour_human_intervention(line_images_ab, tour_5, direction_5)


In [ ]:
# 问题 3 附件 5 行间拼接人工干预

direction_5 = ~(direction_5 ^ parse_direction_string("11111111000"))
problem_3_tour_human_intervention(line_images_ab, tour_5, direction_5)
